# Elektrikli Araç Bataryalarında Çok Amaçlı Optimal Şarj Stratejisinin Nümerik Optimizasyonu — Tam Proje (Tek Parça Google Colab)

* **Ders:** MAT353 – Nümerik Analiz (2026–2027 Güz Dönemi)
* **Bölüm:** Bilgisayar Mühendisliği – İstanbul Medeniyet Üniversitesi
* **Öğretim Üyesi:** Doç. Dr. Murat GEZER
* **Öğrenci:** Samet Oymak (`24120205042`)
* **GitHub Deposu:** https://github.com/Sametoymak99/EV-Battery-Optimization

---

Bu tek parça çalışma defteri projenin **12 aşamasının tamamını** uçtan uca tek seferde çalıştırır (~4–5 dakika):
1. **Aşama 1–2:** Stanford INR21700-M50T ve NASA 18650 şema özetleri, temizleme raporları, OCV/protokol/termal grafikler
2. **Aşama 3–4:** 1-RC Thevenin + birleştirilmiş termal + yaşlanma modeli, analitik doğrulamalar, sentetik HPPC parametre geri kazanımı, bağımsız doğrulama (`W5`, `V4`)
3. **Aşama 5:** İleri Euler, RK4 ve `solve_ivp` yakınsama mertebesi (`DOP853` referanslı), kararlılık sınırı (`120.3 s`) ve sonlu fark gradyan hassasiyeti
4. **Aşama 6:** Geleneksel CC-CV referans taraması (`0.5C – 2.0C`) ve `1.0C` normalizasyon ölçekleri
5. **Aşama 7–9:** Gradient Descent (Armijo + 2-kademeli ceza), Adam (kosinüs LR) ve SciPy SLSQP optimizasyonu + çoklu-seed (`42, 123, 2026`)
6. **Aşama 10:** 20 ağırlık üçlüsüyle çok amaçlı Pareto cephesi analizi
7. **Aşama 11:** Sabit tohumlu (`seed=2026`) 200 örneklemli Monte Carlo belirsizlik ve Pearson duyarlılık analizi
8. **Aşama 12:** Otomatik `pytest` birim ve entegrasyon test paketi

In [ ]:
# 1. Ortam Kurulumu ve GitHub Deposunun Klonlanması (Colab + Yerel Uyumlu)
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Sametoymak99/EV-Battery-Optimization.git'

if IN_COLAB:
    target_dir = Path('/content/EV-Battery-Optimization')
    if not target_dir.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(target_dir)], check=True)
    else:
        subprocess.run(['git', '-C', str(target_dir), 'pull', '--ff-only'], check=False)
    os.chdir(target_dir)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
PROC_DIR = PROJECT_ROOT / 'data' / 'processed'
TABLES_DIR = PROJECT_ROOT / 'results' / 'tables'
FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures'
OUTPUTS_DIR = PROJECT_ROOT / 'results' / 'simulation_outputs'

import numpy as np
import pandas as pd
import scipy
import matplotlib
import matplotlib.pyplot as plt
try:
    from IPython.display import Image, display
except ImportError:
    def Image(filename=None):
        return f'[Görsel: {filename}]'
    def display(obj):
        print(obj.to_string(index=False) if hasattr(obj, 'to_string') else obj)

print(f'Çalışma ortamı : {"Google Colab" if IN_COLAB else "Yerel Python"}')
print(f'Proje kökü     : {PROJECT_ROOT}')
print(f'Python / NumPy / SciPy / Pandas: {sys.version.split()[0]} / {np.__version__} / {scipy.__version__} / {pd.__version__}')

src_files = sorted((PROJECT_ROOT / 'src').glob('*.py'))
total_lines = 0
print('\n--- Çekirdek Modüller (src/) ---')
for fp in src_files:
    n_lines = len(fp.read_text(encoding='utf-8').splitlines())
    total_lines += n_lines
    print(f'  {fp.name:<28s} {n_lines:4d} satır')
print(f'Toplam src/ kodu: {total_lines} satır')

---
## Aşama 1 & 2: Veri Setleri, Ön İşleme ve Fiziksel Ayrım
* **Ana Veri Seti:** Stanford Energy Control Laboratory — LG Chem `INR21700-M50T` (`https://osf.io/qsabn/`)
  * **Kalibrasyon / Eğitim:** `W4` (C/4), `W8` (C/2), `W9` (1C), `W10` (3C) `Diag_1` + `W10_Channel_6.2` termal verisi
  * **Bağımsız Doğrulama:** `W5` (C/2) ve `V4` (C/4) *(kalibrasyona hiç dahil edilmez)*
* **Yardımcı Veri Seti:** NASA Ames PCoE `18650` (`B0005, B0006, B0007, B0018, B0029, B0053`) — farklı hücre kimyası/form faktörü nedeniyle Stanford verisiyle **birleştirilmez**.
* *Not:* Ham `.mat/.xlsx` dosyaları boyut (~760 MB) ve lisans nedeniyle depoya konmamıştır; ön işlenmiş gerçek eğriler `data/processed/` altındadır (tam ham veri indirme adımları `data/README.md` dosyasındadır).

In [ ]:
# Aşama 1 & 2: Şema Özetleri, Temizleme Raporu ve Gerçek Deney Grafikleri
stanford_schema = pd.read_csv(TABLES_DIR / 'stanford_schema_summary.csv')
nasa_schema = pd.read_csv(TABLES_DIR / 'nasa_schema_summary.csv')
clean_rep = pd.read_csv(TABLES_DIR / 'phase2_cleaning_report.csv')

print('=== Stanford Şema Özeti (İlk 8 Satır) ===')
display(stanford_schema.head(8))
print('=== NASA Şema Özeti ===')
display(nasa_schema)
print('=== Aşama 2 Temizleme Raporu ===')
display(clean_rep)

ocv_df = pd.read_csv(PROC_DIR / 'stanford_ocv_c20_diag1.csv')
chg_df = pd.read_csv(PROC_DIR / 'stanford_rep_charge_profiles.csv')
therm_df = pd.read_csv(PROC_DIR / 'stanford_w10_thermal_sample.csv')
nasa_df = pd.read_csv(PROC_DIR / 'nasa_rep_charge_cycles.csv')

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for cid in ['W4', 'W8', 'W9', 'W10']:
    sub = ocv_df[ocv_df['cell_id'] == cid]
    axes[0, 0].plot(sub['soc'], sub['voltage_V'], linewidth=1.6, label=cid)
axes[0, 0].set_title('Stanford C/20 Tanı OCV(SOC) Eğrileri')
axes[0, 0].set_xlabel('SOC [-]'); axes[0, 0].set_ylabel('Gerilim [V]')
axes[0, 0].legend(fontsize=8); axes[0, 0].grid(True, alpha=0.3)

for cid in ['W4', 'W8', 'W9', 'W10']:
    sub = chg_df[chg_df['cell_id'] == cid]
    lbl = f"{cid} ({sub['c_rate'].iloc[0]})"
    axes[0, 1].plot(sub['time_s'] / 60.0, sub['voltage_V'], linewidth=1.5, label=lbl)
axes[0, 1].set_title('Stanford Cycling_1 Çevrim-2 Şarj Profilleri')
axes[0, 1].set_xlabel('Süre [dk]'); axes[0, 1].set_ylabel('Terminal Gerilimi [V]')
axes[0, 1].legend(fontsize=8); axes[0, 1].grid(True, alpha=0.3)

ax_t = axes[1, 0]
ax_t.plot((therm_df['time_s'] - therm_df['time_s'].iloc[0]) / 60.0, therm_df['temperature_C'], color='tab:red', linewidth=1.8, label='Sıcaklık [°C]')
ax_t.set_title('Stanford W10 (3C) Yüzey Sıcaklığı (Aux_Temp)')
ax_t.set_xlabel('Süre [dk]'); ax_t.set_ylabel('Sıcaklık [°C]', color='tab:red')
ax_t.grid(True, alpha=0.3)

for cid in sorted(nasa_df['cell_id'].unique()):
    sub = nasa_df[nasa_df['cell_id'] == cid]
    tamb = sub['ambient_temperature_C'].iloc[0]
    axes[1, 1].plot(sub['time_s'] / 60.0, sub['temperature_C'], linewidth=1.5, label=f'{cid} (Tamb={tamb:.0f}°C)')
axes[1, 1].set_title('NASA 18650 Yardımcı Veri Seti Şarj Sıcaklıkları')
axes[1, 1].set_xlabel('Süre [dk]'); axes[1, 1].set_ylabel('Sıcaklık [°C]')
axes[1, 1].legend(fontsize=8); axes[1, 1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

---
## Aşama 3 & 4: Matematiksel Model, Analitik Doğrulama ve Parametre Kestirimi
Durum vektörü $x(t) = [\text{SOC}(t),\; V_p(t),\; T(t),\; D(t)]^T$ ve kontrol değişkeni $u(t) = I(t)$ için birinci dereceden Thevenin eşdeğer devresi + birleştirilmiş termal denklem + ampirik göreli yaşlanma modeli:
$$\frac{d\text{SOC}}{dt} = \frac{\eta\,I(t)}{3600\,Q}, \qquad \frac{dV_p}{dt} = -\frac{V_p}{R_1 C_1} + \frac{I(t)}{C_1}, \qquad V_t = \text{OCV}(\text{SOC}) + I(t) R_0 + V_p$$
$$C_{\text{th}} \frac{dT}{dt} = I(t)^2 R_0 + \frac{V_p^2}{R_1} - \frac{T - T_{\text{amb}}}{R_{\text{th}}}, \qquad \frac{dD}{dt} = k_0 \exp\!\left(-\frac{E_a}{R_g (T+273.15)}\right) \left(\frac{|I(t)|}{Q}\right)^p (1 + \alpha_{\text{soc}}\,\text{SOC}^2)$$

In [ ]:
# Aşama 3 & 4: Kalibre Parametreler, Analitik Kontroller ve Sentetik HPPC Geri Kazanımı
from src.battery_model import (
    BatteryLimits,
    BatteryParameters,
    OCVInterpolator,
    euler_stability_limit,
    simulate_current_profile,
)
from src.parameter_estimation import estimate_electrical_params_hppc, simulate_1rc_voltage

cal_json = PROC_DIR / 'calibrated_parameters.json'
params = BatteryParameters.from_calibrated_json(cal_json)
ocv_fn = OCVInterpolator.from_calibrated_json(cal_json)
limits = BatteryLimits()

print('Kalibre Edilmiş Parametreler:', params)
print(f'Açık Euler Kararlılık Sınırı : {euler_stability_limit(params):.2f} s')

# 1) Analitik RC Polarizasyon Doğrulaması
t_rc = np.linspace(0.0, 180.0, 181)
i_const = 4.0
res_rc = simulate_current_profile(t_rc, np.full_like(t_rc, i_const), params, ocv_fn, method='rk4')
vp_exact = i_const * params.R1_Ohm * (1.0 - np.exp(-t_rc / (params.R1_Ohm * params.C1_F)))
rc_err = float(np.max(np.abs(res_rc['vp_V'] - vp_exact)))
assert rc_err < 1e-6
print(f'Analitik RC Polarizasyon Maks. Hata (RK4) : {rc_err:.3e} V (< 1e-6 V OK)')

# 2) Analitik Termal Soğuma Doğrulaması
t_th = np.linspace(0.0, 1200.0, 601)
x0_cool = np.array([0.50, 0.0, 40.0, 0.0, 0.0])
res_cool = simulate_current_profile(t_th, np.zeros_like(t_th), params, ocv_fn, x0=x0_cool, method='rk4')
t_exact = params.T_amb_C + (40.0 - params.T_amb_C) * np.exp(-t_th / (params.Rth_K_per_W * params.Cth_J_per_K))
cool_err = float(np.max(np.abs(res_cool['temperature_C'] - t_exact)))
assert cool_err < 1e-6
print(f'Analitik Termal Soğuma Maks. Hata (RK4)   : {cool_err:.3e} °C (< 1e-6 °C OK)')

# 3) Canlı Sentetik HPPC Parametre Geri Kazanımı (< %2 Hata)
t_syn = np.arange(0.0, 2000.0, 1.0)
curr_syn = np.zeros_like(t_syn)
curr_syn[100:200] = -4.5; curr_syn[500:550] = 2.0; curr_syn[900:1100] = -4.5; curr_syn[1400:1500] = -3.0
ocv_syn = OCVInterpolator(np.linspace(0.0, 1.0, 21), 3.0 + 1.2 * np.linspace(0.0, 1.0, 21), cell_id='synth')
soc_syn = np.clip(1.0 + np.cumsum(curr_syn) / (3600.0 * 4.85), 0.0, 1.0)
vt_syn, _ = simulate_1rc_voltage(t_syn, curr_syn, ocv_syn(soc_syn), 0.028, 0.014, 1800.0)
rec = estimate_electrical_params_hppc(pd.DataFrame({'time_s': t_syn, 'current_A': curr_syn, 'voltage_V': vt_syn}), ocv_syn, q_ah=4.85, soc_window=(0.10, 0.999))
print(f"Sentetik HPPC Geri Kazanım -> R0={rec['R0_Ohm']:.5f} (0.02800), R1={rec['R1_Ohm']:.5f} (0.01400), C1={rec['C1_F']:.1f} (1800.0)")

print('\n=== Aşama 4 Parametre Kestirim Tablosu ===')
display(pd.read_csv(TABLES_DIR / 'phase4_parameter_estimation.csv'))
print('=== Bağımsız Doğrulama Metrikleri (W5 ve V4 dahil) ===')
display(pd.read_csv(TABLES_DIR / 'phase4_validation_metrics.csv'))

for fn in ['fig04_hppc_calibration_validation.png', 'fig04_thermal_calibration_w10.png']:
    fp = FIGURES_DIR / fn
    if fp.exists():
        display(Image(filename=str(fp)))

---
## Aşama 5: Sayısal ODE Çözücüleri (`Euler`, `RK4`, `solve_ivp`) ve Sonlu Fark Gradyan Analizi
Yüksek mertebeli `DOP853` (`rtol=1e-12`) referansına karşı ampirik yakınsama mertebesi, açık Euler kararlılık sınırı ($\Delta t = 130\text{ s} > 120.30\text{ s}$) ve ileri/merkezi sonlu fark adım boyu ($h$) hassasiyeti canlı çalıştırılır.

In [ ]:
# Aşama 5: Canlı ODE Karşılaştırması ve Sonlu Fark Analizi
from src.numerical_analysis import run_phase5_pipeline

res5 = run_phase5_pipeline(
    processed_dir=PROC_DIR,
    tables_dir=TABLES_DIR,
    figures_dir=FIGURES_DIR,
)
print('Ampirik Yakınsama Mertebeleri:', res5['convergence_orders'])
display(res5['ode_benchmark'])
display(res5['soc_dt_sensitivity'])
display(res5['fd_sensitivity'])
for fp in res5['saved_figures']:
    display(Image(filename=fp))

---
## Aşama 6: Geleneksel CC-CV Referans Şarj Stratejisi (`0.5C – 2.0C`)
`SOC: 0.20 → 0.90`, $V_{\max} = 4.20\text{ V}$ altında `0.5C, 0.7C, 1.0C, 1.5C, 2.0C` CC-CV profilleri canlı simüle edilir ve çok amaçlı normalizasyon için `1.0C` referans ölçekleri $(t_{\text{ref}}, E_{\text{ref}}, D_{\text{ref}})$ hesaplanır.

In [ ]:
# Aşama 6: Canlı CC-CV Taraması ve Referans Ölçekler
from src.cc_cv import run_phase6_cc_cv_benchmark

res6 = run_phase6_cc_cv_benchmark(
    processed_dir=PROC_DIR,
    tables_dir=TABLES_DIR,
    figures_dir=FIGURES_DIR,
)
print('1.0C Referans Ölçekler:', res6['reference_scales'])
display(res6['comparison_table'])
display(Image(filename=res6['figure_path']))

---
## Aşama 7, 8 & 9: Gradient Descent, Adam ve SciPy SLSQP Optimizasyonu + Çoklu-Seed Duyarlılığı
$N=10$ eşit $\Delta\text{SOC}=0.07$ dilimli akım vektörü $\mathbf{u}=[I_1,\dots,I_{10}]^T$ üzerinde dengeli ağırlıklar $w = [0.4, 0.2, 0.4]$ ile üç algoritma ve 3 farklı başlangıç tohumu (`seed = 42, 123, 2026`) canlı çalıştırılır.

In [ ]:
# Aşama 7-9: Canlı Optimizasyon Karşılaştırması ve Çoklu-Seed Testi
from src.optimizers import run_phases_7_to_9_experiments

exp9 = run_phases_7_to_9_experiments(
    processed_dir=PROC_DIR,
    tables_dir=TABLES_DIR,
    figures_dir=FIGURES_DIR,
    outputs_dir=OUTPUTS_DIR,
    weights=(0.4, 0.2, 0.4),
    seeds=(42, 123, 2026),
)
print('=== Algoritma Karşılaştırması (w=[0.4, 0.2, 0.4]) ===')
display(exp9['comparison_table'])
print('=== Çoklu Başlangıç Tohumu Duyarlılığı (seed=42, 123, 2026) ===')
display(exp9['multiseed_table'])

for fn in ['fig07_gradient_descent.png', 'fig08_adam_optimization.png', 'fig09_optimizer_comparison.png']:
    fp = FIGURES_DIR / fn
    if fp.exists():
        display(Image(filename=str(fp)))

---
## Aşama 10: 20 Ağırlıklı Çok Amaçlı Pareto Cephesi Analizi
Farklı ağırlık üçlüleri $(w_1, w_2, w_3)$ ($\sum w_i = 1$) taranarak hız, enerji kaybı ve yaşlanma arasındaki Pareto-optimal ödünleşim cephesi canlı hesaplanır ve `0.5C–2.0C` CC-CV eğrisiyle kıyaslanır.

In [ ]:
# Aşama 10: Canlı Pareto Analizi (20 Ağırlık Senaryosu)
from src.evaluation import run_pareto_analysis

res10 = run_pareto_analysis(
    processed_dir=PROC_DIR,
    tables_dir=TABLES_DIR,
    figures_dir=FIGURES_DIR,
    outputs_dir=OUTPUTS_DIR,
)
print(f"Uygun çözüm: {res10['n_feasible']}, Pareto-optimal: {res10['n_pareto_optimal']}")
display(res10['pareto_table'])
display(Image(filename=res10['figure_path']))

---
## Aşama 11: Monte Carlo Parametre Belirsizlik Analizi ($N=200$, `seed=2026`)
Kalibrasyon belirsizlikleri $(\sigma_Q, \sigma_{R_0}, \sigma_{R_1}, \sigma_{C_1}, \sigma_{R_{\text{th}}}, \sigma_{C_{\text{th}}})$ altında açık döngü SLSQP optimal profilinin ve kapalı döngü `1.0C` CC-CV stratejisinin performansı ile kısıt aşım olasılıkları canlı test edilir.

In [ ]:
# Aşama 11: Canlı Monte Carlo Simülasyonu (N=200, seed=2026)
from src.evaluation import run_monte_carlo_analysis

res11 = run_monte_carlo_analysis(
    processed_dir=PROC_DIR,
    tables_dir=TABLES_DIR,
    figures_dir=FIGURES_DIR,
    outputs_dir=OUTPUTS_DIR,
    n_samples=200,
    seed=2026,
)
print('=== Monte Carlo Özet Tablosu ===')
display(res11['summary_table'])
print('=== Pearson Korelasyon Duyarlılık Analizi ===')
display(res11['sensitivity_table'])
display(Image(filename=res11['figure_path']))

---
## Aşama 12: Otomatik `pytest` Birim ve Entegrasyon Test Paketi
Ham Stanford `.mat` dosyaları yerelde mevcutsa **tüm 8 test dosyası (28 test)**, Colab ortamında ham `.mat` dosyaları olmadan çalıştırılıyorsa model/ön-işleme/kalibrasyon/ODE/CC-CV/optimizasyon/değerlendirme modüllerini kapsayan **21 Colab-uyumlu test** otomatik çalıştırılıp doğrulanır.

In [ ]:
# Aşama 12: Canlı Pytest Doğrulaması
raw_cap = DATA_RAW / 'stanford' / 'diagnostic' / 'capacity_test.mat'
if raw_cap.exists():
    cmd = [sys.executable, '-m', 'pytest', 'tests/', '-v']
    print('Ham veri mevcut -> Tüm 28 test çalıştırılıyor...')
else:
    cmd = [
        sys.executable, '-m', 'pytest',
        'tests/test_battery_model.py',
        'tests/test_parameter_estimation.py',
        'tests/test_numerical_analysis.py',
        'tests/test_cc_cv.py',
        'tests/test_optimizers.py',
        'tests/test_evaluation.py',
        '-v',
    ]
    print('Colab ortamı (ham .mat olmadan) -> 19 çekirdek model/optimizasyon testi çalıştırılıyor...')

proc = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True)
print(proc.stdout)
if proc.stderr:
    print(proc.stderr)
assert proc.returncode == 0, f'Pytest hata koduyla sonlandı: {proc.returncode}'
print('TÜM TESTLER BAŞARIYLA GEÇTİ!')